In [1]:
import os
from pyspark.sql import SparkSession

# Stop session sebelumnya biar port 15002 lepas
try:
    spark.stop()
    print("🛑 Spark session sebelumnya dihentikan")
except:
    pass

def create_spark_session(env="local"):
    """
    Spark Session adaptif untuk:
    - 'local'  : MinIO (development/local testing)
    - 'aws'    : AWS S3 (production via Terraform)
    """
    # Package AWS untuk Hadoop 3.3.x
    aws_packages = "org.apache.hadoop:hadoop-aws:3.4.2,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.postgresql:postgresql:42.7.3"
    builder = SparkSession.builder \
        .appName("Franchise-Data-Pipeline") \
        .config("spark.driver.memory", "2g") \
        .config("spark.jars.packages", aws_packages) \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.skewJoin.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.hadoop.fs.s3a.multipart.size", "104857600") \
        .config("spark.hadoop.fs.s3a.threads.max", "20") \
        .config("spark.hadoop.fs.s3a.committer.name", "magic") \
        .config("spark.plugins", "org.apache.spark.sql.connect.SparkConnectPlugin") \
        .config("spark.connect.grpc.binding.port", "15002")
    
    # Kustomisasi per environment
    if env == "local":
        print("🔧 LOCAL: Connecting to MinIO at localhost:9000")
        builder = builder \
            .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
            .config("spark.hadoop.fs.s3a.path.style.access", "true") \
            .config("spark.hadoop.fs.s3a.aws.credentials.provider", 
                    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
            .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ACCESS_KEY", "minioadmin")) \
            .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_SECRET_KEY", "minioadmin"))
            
    elif env == "aws":
        print("☁️ AWS: Connecting to S3 with IAM Role (via Terraform)")
        builder = builder \
            .config("spark.hadoop.fs.s3a.path.style.access", "false") \
            .config("spark.hadoop.fs.s3a.aws.credentials.provider", 
                    "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
            .config("spark.hadoop.fs.s3a.endpoint", "")
    
    return builder.getOrCreate()

# =========================================================================
# PENGGUNAAN
# =========================================================================
RUNNING_ENV = "local"
spark = create_spark_session(env=RUNNING_ENV)

print("✅ Spark session nyala di port 15002")
print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")


🔧 LOCAL: Connecting to MinIO at localhost:9000


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/27 14:00:34 WARN Utils: Your hostname, sanju, resolves to a loopback address: 127.0.1.1; using 192.168.1.77 instead (on interface wlp0s20f3)
26/05/27 14:00:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/sanju/miniconda3/envs/basic_dev/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/sanju/.ivy2.5.2/cache
The jars for the packages stored in: /home/sanju/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-aa847908-3b05-45da-aa86-f6cc3e8640f7;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central

✅ Spark session nyala di port 15002
Spark Version: 4.1.1
Spark Master: local[*]


SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".    (0 + 1) / 1]
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [4]:
# Test koneksi
try:
    if RUNNING_ENV == "local":
        spark.read.csv("s3a://franchise-pipeline-data-lake-bronze/order_items/year=2026/month=03/day=01/order_items.csv").show(5)
    else:
        spark.read.csv("s3a://franchise-pipeline-dev-data-lake-bronze/order_items/year=2026/month=03/day=02/order_items.csv").show(5)
    print("✅ Connection successful!")
except Exception as e:
    print(f"❌ Connection failed: {e}")

+-------+--------+-------+--------+--------------+--------+
|    _c0|     _c1|    _c2|     _c3|           _c4|     _c5|
+-------+--------+-------+--------+--------------+--------+
|item_id|order_id|menu_id|quantity|price_per_item|subtotal|
|2517633| 1045592|      3|       1|      22000.00|22000.00|
|2482817| 1031165|     11|       2|      39000.00|78000.00|
|2482818| 1031165|     20|       2|      25000.00|50000.00|
|2482819| 1031165|     30|       3|      26000.00|78000.00|
+-------+--------+-------+--------+--------------+--------+
only showing top 5 rows
✅ Connection successful!
